### Import Modules/Data

#### Imports

In [ ]:
from dask.distributed import Client

client = Client()  # your client

# Show basic client info
print(client)

# More detailed info about workers
for w in client.scheduler_info()['workers'].values():
    print(f"Worker: {w['name']}, Threads: {w['nthreads']}, Memory limit: {w['memory_limit']}")

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as ss
import dask
import matplotlib.pyplot as plt
import xarray as xr              #library to read in and process Earth system data in NetCDF files
import cartopy.crs as ccrs       #library to handle maps
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import os
import cartopy.mpl.ticker as cticker
import netCDF4
import glob
import re

from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
from matplotlib.patches import Patch
import matplotlib.colors as colors

import matplotlib
from IPython.display import HTML, display
from pathlib import Path

# Point matplotlib to the binary
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
print("ffmpeg path set to:", imageio_ffmpeg.get_ffmpeg_exe())
from pandas.plotting import register_matplotlib_converters #function used in plotting when converting NetCDF data to dates 
register_matplotlib_converters() #needed to facilitate conversion of dates in netCDF files
%matplotlib inline

#### Dataset File Load-in

In [ ]:
# Enter Directory 
your_dir = "path/to/directory"

import sys
sys.path.append(your_dir +'/.local/lib/python3.10/site-packages')

In [ ]:
#MERRA2 DATA
path = your_dir + "/data/MERRA2 Data"

# Gather files
files_nc4 = sorted(glob.glob(os.path.join(path, "*.nc4")))
files_nc  = sorted(glob.glob(os.path.join(path, "*.nc")))

chunks = {"time": 20, "lat": 60, "lon": 120}
# Open with fast h5netcdf backend
MERRAds1 = xr.open_mfdataset(files_nc4,combine="nested",concat_dim="time",engine="h5netcdf",chunks=chunks,parallel=True,)
MERRAds2 = xr.open_mfdataset(files_nc,combine="by_coords",engine="h5netcdf",chunks=chunks,parallel=True,)
MERRAds = xr.combine_nested([MERRAds1, MERRAds2], concat_dim="time")
print(MERRAds.dims, MERRAds.data_vars)

encoding = {}
for var in MERRAds.data_vars:
    da = MERRAds[var]
    if hasattr(da.data, "chunks"):
        # must match dimension order exactly
        chunksizes = tuple(c[0] for c in da.data.chunks )  
        encoding[var] = {"chunksizes": chunksizes}


In [ ]:
#ERA5
path = your_dir + r"/data/ERA5 Data"
ERA5presds = xr.open_dataset(path + r"/f52fd0451e6d1374511b6b2dd0404cad.nc")
ERA5surfds = xr.open_dataset(path + r"/6a5a721e6db7f9237dfb12915a9f6546.nc")
ERA5presds = ERA5presds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ERA5surfds = ERA5surfds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
print(ERA5surfds.dims, ERA5surfds.data_vars)
print(ERA5presds.dims, ERA5presds.data_vars)

In [ ]:
#AIRS
path = your_dir+ r"/data/AIRS V7 New"
pattern = os.path.join(path, "*.hdf.nc4")
files = sorted(glob.glob(pattern))

datasets = []
for f in files:
    match = re.search(r"AIRS\.(\d{4})\.(\d{2})\.(\d{2})", os.path.basename(f))
    year, month, day = map(int, match.groups())
    date = pd.Timestamp(year, month, day)

    ds = xr.open_dataset(f, engine="netcdf4", decode_timedelta=True)
    ds = ds.expand_dims(dim={"time": [date]})
    datasets.append(ds)

AIRS7ds = xr.concat(datasets, dim="time")
AIRS7ds = AIRS7ds.rename({'Latitude': 'lat', 'Longitude': 'lon'})
print(AIRS7ds.dims, AIRS7ds.data_vars)

In [ ]:
# Berkeley Earth
Berkds = xr.open_dataset(your_dir +r'/data/Berkeley Earth/Land_and_Ocean_LatLong1.nc')

# Convert fractional years to datetime for temperature
Berkds['time'] = pd.to_datetime((Berkds['time'].values - 1970) * 365.25, unit='D', origin='1970-01-01')
Berkds = Berkds.rename({'latitude': 'lat', 'longitude': 'lon'})
# Extract months from temperature time
temp_months = Berkds['time'].dt.month - 1  # 0-based index for climatology
# Expand climatology to match temperature time
clim_expanded = Berkds['climatology'].isel(month_number=temp_months)
# Assign the same time coordinate as temperature
clim_expanded = clim_expanded.assign_coords(time=Berkds['time'])
# Compute absolute temperature
Berkds['true_temperature'] = Berkds['temperature'] + clim_expanded

print(Berkds.dims, Berkds.data_vars)
print(Berkds['time'].values)
print(Berkds['lat'].values, Berkds['lon'].values)

In [ ]:
#CLIMCAPS Data
path = your_dir+ r"/data/CLIMCAPS Data/Surface2"
pattern = os.path.join(path, "*.nc4")
datasets = []
files = sorted(glob.glob(pattern))
for f in files:
    year, month, day = map(int, re.search(r"SNDR.AQUA.AIRS\.(\d{4})(\d{2})(\d{2})", os.path.basename(f)).groups())
    date = pd.Timestamp(year, month, day)
    ds = xr.open_dataset(f, engine="netcdf4", decode_timedelta=True)
    ds = ds.expand_dims(dim={"time": [date]})
    datasets.append(ds)
CLIMCAPSds = xr.concat(datasets, dim="time")
CLIMsurfds = CLIMCAPSds.mean(dim='orbit_pass')

In [ ]:
#JP55 surface data
def open_concat(files, varname_hint=None):
    datasets = []
    for f in files:
        ds = xr.open_dataset(f, engine='cfgrib', decode_timedelta=True)
        datasets.append(ds)
    return xr.concat(datasets, dim='time')

path = your_dir+ r"/data/JP55 Data/Surface"
for f in glob.glob(os.path.join(path, "*.idx")):
    os.remove(f)

temp_files = sorted(glob.glob(os.path.join(path, "anl_surf.*_tmp.*")))
humid_files = sorted(glob.glob(os.path.join(path, "anl_surf.*_spfh.*")))

tempsurf_ds = open_concat(temp_files)
humidsurf_ds = open_concat(humid_files)
JRA55surfds = xr.merge([tempsurf_ds, humidsurf_ds], compat='override')

#Pressure data
path = your_dir+ r"/data/JP55 Data/500_hPa"
for f in glob.glob(os.path.join(path, "*.idx")):
    os.remove(f)
temppres500_files = sorted(glob.glob(os.path.join(path, "anl_p125.*_tmp.*")))

path = your_dir+ r"/data/JP55 Data/850_hPa"
for f in glob.glob(os.path.join(path, "*.idx")):
    os.remove(f)
temppres850_files = sorted(glob.glob(os.path.join(path, "anl_p125.*_tmp.*")))

temppres500ds = open_concat(temppres500_files)
temppres850ds = open_concat(temppres850_files)
JRA55presds = xr.concat([temppres850ds, temppres500ds], dim="isobaricInhPa")
JRA55presds = JRA55presds.rename({'latitude': 'lat', 'longitude': 'lon'})
JRA55surfds = JRA55surfds.rename({'latitude': 'lat', 'longitude': 'lon'})

print(JRA55surfds.dims, JRA55surfds.data_vars)
print(JRA55presds.dims, JRA55presds.data_vars)

In [ ]:
#JRA-3Q surface data
def open_concat(files):
    return xr.open_mfdataset(files,combine="nested", concat_dim="time",engine="netcdf4")

path = your_dir+ r"/data/JRA3Q Data/Surface"
temp_files = sorted(glob.glob(os.path.join(path, "*.anl_surf125.0_0_0.tmp2m-hgt-an-ll125-mn.*.nc"))) 
humid_files = sorted(glob.glob(os.path.join(path, "jra3q-ms-mn.anl_surf125.0_1_0.spfh2m-hgt-an-ll125-mn.*.nc")))

tempsurf_ds = open_concat(temp_files)
humidsurf_ds = open_concat(humid_files)
JRA3Qsurfds = xr.merge([tempsurf_ds, humidsurf_ds], compat='override')
JRA3Qsurfds = JRA3Qsurfds.rename({'tmp2m-hgt-an-ll125-mn': 't2m', 'spfh2m-hgt-an-ll125-mn': 'sh2'})

#Pressure data
path = your_dir+ r"/data/JRA3Q Data/500hPa"
for f in glob.glob(os.path.join(path, "*.idx")):
    os.remove(f)
temppres500_files = sorted(glob.glob(os.path.join(path, "*.tmp-pres-an-ll125-mn.jra3q-ms-mn.anl_p125.0_0_0.tmp-pres-an-ll125-mn.*.nc")))

path = your_dir+ r"/data/JRA3Q Data/850hPa"
for f in glob.glob(os.path.join(path, "*.idx")):
    os.remove(f)
temppres850_files = sorted(glob.glob(os.path.join(path, "*.tmp-pres-an-ll125-mn.jra3q-ms-mn.anl_p125.0_0_0.tmp-pres-an-ll125-mn.*.nc")))

temppres500ds = open_concat(temppres500_files)
temppres850ds = open_concat(temppres850_files)
JRA3Qpresds = xr.concat([temppres850ds, temppres500ds], dim="pressure_level")
JRA3Qpresds = JRA3Qpresds.rename({'tmp-pres-an-ll125-mn': 't'})
print(JRA3Qsurfds)
print(JRA3Qpresds)

In [ ]:
#GISTEMP Data
gistemp = xr.open_dataset(your_dir + '/data/GISTEMP/gistemp1200_GHCNv4_ERSSTv5.nc.gz')
print(gistemp)
GISTEMPds = gistemp['tempanomaly']

#####

#### Time Slicing / Adjustments

In [ ]:
#Check latest time
datasets = [MERRAds, ERA5presds, ERA5surfds, AIRS7ds,
            JRA55presds, JRA55surfds, JRA3Qsurfds, JRA3Qpresds,
            Berkds, CLIMsurfds, GISTEMPds]

sorted_data = []

for ds in datasets:
    time_index = ds.get_index("time")
    if not time_index.is_monotonic_increasing:
        ds = ds.sortby("time")
    print(ds['time'].values.min(), ds['time'].values.max())
    sorted_data.append(ds)

(MERRAds, ERA5presds, ERA5surfds, AIRS7ds,
 JRA55presds, JRA55surfds,JRA3Qsurfds, JRA3Qpresds,
 Berkds, CLIMsurfds, GISTEMPds) = sorted_data


In [ ]:
#TIME SLICER
uncut_data = [MERRAds, ERA5presds, ERA5surfds, AIRS7ds, JRA55presds, JRA55surfds, JRA3Qsurfds, JRA3Qpresds, Berkds, CLIMsurfds]
cut_data = []

for ds in uncut_data:
    ds = ds.sel(time=slice("2002-09-01", "2023-08-31"))
    cut_data.append(ds)
    
(MERRAds, ERA5presds, ERA5surfds, AIRS7ds, JRA55presds, JRA55surfds, JRA3Qsurfds, JRA3Qpresds, Berkds, CLIMsurfds) = cut_data
#cut_data = dask.persist(*cut_data, scheduler='threads')

JRA55presds = JRA55presds.sel(time=slice("2002-09-01", "2023-08-31"))
JRA55surfds = JRA55surfds.sel(time=slice("2002-09-01", "2023-08-31"))
GISTEMPds = GISTEMPds.sel(time=slice("2002-09-01", "2023-08-31"))

### Plotting Helper Functions

In [ ]:
def SixteenPlotter(
        temp,          # list of 16 DataArrays
        pvals,         # list of 16 DataArrays or None
        cmapz,         # list of 16 colormap names
        suptitle,      # figure title
        title,         # list of 16 subplot titles
        vmins,         # list of 16 vmin values (or None)
        vmaxs,         # list of 16 vmax values (or None)
        filename,      # output filename
        variable_name, # colorbar label
        significance=0.05
    ):
    trans = ccrs.PlateCarree()  # Define projection once
    fig, axes = plt.subplots(4, 4, figsize=(20, 10), subplot_kw={'projection': trans}, constrained_layout=True)

    contour_set = None
    grey_map = colors.ListedColormap(['grey'])

    # Loop through 16 panels
    for i, ax in enumerate(axes.flat):
        if temp[i] is None:
            ax.set_visible(False)
            continue

        da = temp[i]
        pval_i = pvals[i] if pvals is not None else None

        # Downsample if needed
        while len(da.lat) > 400 and len(da.lon) > 800:
            da = da.isel(lat=slice(None, None, 2), lon=slice(None, None, 2))
            if pval_i is not None: pval_i = pval_i.isel(lat=slice(None, None, 2), lon=slice(None, None, 2))

        # vmin/vmax per panel — force None → symmetric around 0
        if vmins[i] is None or vmaxs[i] is None:
            absmax = np.nanmax(np.abs(da))
            vmin, vmax = -absmax, absmax
        else:
            vmin, vmax = vmins[i], vmaxs[i]

        # Colormap per panel
        cmap = plt.colormaps[cmapz[i]] if isinstance(cmapz[i], str) else cmapz[i]
        cmap = cmap.copy()
        cmap.set_bad(color='grey')

        # Main contour plot
        contour_set = da.plot.contourf(ax=ax, transform=trans, cmap=cmap, levels=31, vmin=vmin, vmax=vmax, add_colorbar=False)

        # NaN mask (grey)
        nan_mask = da.isnull().astype(float).where(lambda x: x == 1)
        ax.pcolormesh(da['lon'], da['lat'], nan_mask, cmap=grey_map, transform=trans)

        # Stippling for significance
        if pval_i is not None:
            hatch_numeric = np.where(pval_i < significance, 1, np.nan)
            ax.contourf(da['lon'], da['lat'], hatch_numeric, levels=[0.5, 1.5], colors='none', hatches=['////'], transform=trans, antialiased=False)

        # Coastlines, ticks, title
        ax.coastlines()
        ax.set_xticks(np.arange(-180, 181, 60), crs=trans)
        ax.set_yticks(np.arange(-90, 91, 30), crs=trans)
        ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        ax.set_title(title[i], fontsize=12)

    # Colorbar
    if contour_set is not None:
        vmax_list = [v for v in vmaxs if v is not None]
        fmt = '%.2e' if len(vmax_list) == 0 else ('%.2e' if np.nanmax(vmax_list) < 0.04 else '%.2f')

        cbar = fig.colorbar(contour_set, ax=axes, location='right', fraction=0.05, pad=0.02)
        cbar.set_label(variable_name)
        cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(fmt))

    fig.suptitle(suptitle, fontsize=16)
    fig.savefig(filename, dpi=400, bbox_inches='tight')
    plt.close(fig)

In [ ]:
def SixPlotter67(temp, cmap, suptitle, title, vmin, vmax, filename, variable_name, pvals=None, significance=0.05):

    if isinstance(cmap, str): cmap = plt.colormaps[cmap]
    cmap = cmap.copy()
    cmap.set_bad(color='none')  # NaNs handled manually now
    
    by = int(np.ceil(len(temp) / 2))
    proj = ccrs.PlateCarree()
    
    fig, axes = plt.subplots(by, 2, figsize=(10, 2.5 + 2.5 * by), subplot_kw={'projection': proj}, constrained_layout=True)

    contour_set = None
    levels = np.linspace(vmin, vmax, 31)
    grey_map = colors.ListedColormap(['grey'])

    for i, ax in enumerate(axes.flat):
        if temp[i] is None:
            ax.set_visible(False)
            continue

        da = temp[i]

        # --- MAIN FIELD ---
        contour_set = ax.contourf(da['lon'], da['lat'], da, levels=levels, cmap=cmap, extend='both', transform=proj)

        # --- GREY NaN MASK ---
        nan_mask = da.isnull().astype(float).where(lambda x: x == 1)
        ax.pcolormesh(da['lon'], da['lat'], nan_mask, cmap=grey_map, transform=proj)

        # --- SIGNIFICANCE HATCHING ---
        if pvals is not None and pvals[i] is not None:
            hatch_numeric = np.where(pvals[i] < significance, 1, np.nan)
            ax.contourf(da['lon'], da['lat'], hatch_numeric, levels=[0.5, 1.5], colors='none', hatches=['////'], transform=proj, antialiased=False)

        # --- MAP STYLING ---
        ax.coastlines()
        ax.set_xticks(np.arange(-180, 181, 60), crs=proj)
        ax.set_yticks(np.arange(-90, 91, 30), crs=proj)
        ax.set_title(title[i])
        ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

    # --- COLORBAR ---
    if contour_set is not None:
        cbar = fig.colorbar(contour_set, ax=axes, location='bottom', fraction=0.04, pad=0.03, orientation='horizontal')
        cbar.ax.tick_params(labelsize=11)
        cbar.set_label(variable_name, fontsize=13, labelpad=12)
        cbar.locator = mticker.MaxNLocator(nbins=10)
        cbar.update_ticks()
        cbar.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

    fig.suptitle(suptitle, fontsize=19, y=1.04)
    fig.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close(fig)

### Loading Variables (SAT) and Climatologies

In [ ]:
#Determine the trends for surface temperature
timez = [ AIRS7ds['time'], CLIMsurfds['time'], MERRAds['time'], ERA5surfds['time'], JRA55surfds['time'], Berkds['time'], JRA3Qsurfds['time'], GISTEMPds['time'], AIRS7ds['time']]
surftempz = [(AIRS7ds['SurfAirTemp_A']+AIRS7ds['SurfAirTemp_D'])/2, CLIMsurfds['surf_air_temp'], MERRAds['T2M'], ERA5surfds['t2m'], JRA55surfds['t2m'], Berkds['true_temperature']+273.15, JRA3Qsurfds['t2m'], GISTEMPds, (AIRS7ds['SurfSkinTemp_A']+AIRS7ds['SurfSkinTemp_D'])/2]

#var_arr = [surftempz, spechumz, tempz850, tempz500]
var_arr = [surftempz, None, None, None]

In [ ]:
def normalize_lon(da):
    da = da.assign_coords(lon=((da.lon + 180) % 360) - 180)
    da = da.sortby("lon")
    return da

for j in range(0,1):
    for i in range(0,7):
        if var_arr[j][i] is None:
            continue
        var_arr[j][i] = normalize_lon(var_arr[j][i])

In [ ]:
#21-year Mean Climatology (2002-2023)

tru_meanz =  np.empty((4,9), dtype='object')

for j in range(0,1):
    for i in range(0,9):
        if var_arr[j][i] is None:
            tru_meanz[j][i] = None
            continue
        tru_meanz[j][i] = var_arr[j][i].mean(dim='time',skipna=True)
        if j == 1:
            tru_meanz[j][i] = 1000* tru_meanz[j][i]
    print(f'Weighted Mean for Variable {j} is complete!')

In [ ]:
sourcelabels = ['AIRSv7',  'CLIMCAPS-AQUA', 'MERRA2',  'ERA5', 'JRA55', 'JRA3Q','Berkeley Earth', 'AIRSv7 Skin']
varlabels=['Surface Temperature', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [g/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST_mean_clim','SH_mean_clim','850T_mean_clim','500T_mean_clim']
vextremas = [1, 0.001*1000, 1, 1]
colormapz = ['YlOrRd', 'GnBu', 'YlOrRd', 'YlOrRd']


plt.close('all')

for j in range(0,1):  # loop over variables
    order = [0, 1, 2, 3, 4, 5, 6, 8]
    tru_meanz_splice = [tru_meanz[j][i] for i in order]
    # 6-panel plotting
    SixPlotter67(
        temp=tru_meanz_splice,
        cmap=colormapz[j],
        suptitle=f'Global {varlabels[j]} Mean Climatology (Sep 2002 – Aug 2023)',
        title=sourcelabels,
        vmin=220,
        vmax=300,
        filename='New_'+filelabels[j]+'.png',
        variable_name=unitlabels[j],
        pvals=None,
        significance=0.05
    )

### Global Gridded Multiple Linear Regression

##### Import Climate Components to Regress Against

In [ ]:
#import SAOD dataset (this is the only special case, since we manually calculate and save the global mean)
saodDS_total = xr.open_dataset(your_dir + r'/Fitting Components/GloSSAC_V2.23_NC4.nc')

GLOCCASSC = saodDS_total['Glossac_Aerosol_Optical_Depth'].sel(wavelengths_glossac=525) # using 525 nm SAOD
GLOCCASSC["time"] = pd.to_datetime(GLOCCASSC["time"].astype(str), format="%Y%m")

var_z = GLOCCASSC.sortby('lat').sortby('time').persist()
weights = np.cos(np.deg2rad(var_z['lat']))
glob_mean = var_z.weighted(weights).mean(dim=('lat')).persist()
month_avg = glob_mean.groupby('time.month').mean(dim='time')
tot_avg = glob_mean.mean(dim='time').compute().item()
SAODds = glob_mean.groupby('time.month') - month_avg
SAODds = SAODds.sel(time=slice('2002-09',None))
print(SAODds)

SAODds.to_netcdf("global_saod.nc")

In [ ]:
#Import Datasets Function

def open_sesame(filename):
    path = your_dir + r'/Fitting Components//'
    with open(path+filename+'.txt', "r") as f:
        lines = f.readlines()
    # Skip the first header line
    data_lines = [line for line in lines if line.strip() and line.strip()[0].isdigit()]
    # Parse year + 12 monthly values
    records = []
    for line in data_lines:
        parts = line.strip().split()
        if len(parts) == 13:  # year + 12 months
            year = int(parts[0])
            monthly_vals = [float(val) for val in parts[1:]]
            for month, val in enumerate(monthly_vals, 1):
                records.append((f"{year}-{month:02d}", val))
    
    # Convert to x-array
    df = pd.DataFrame(records, columns=["time", "amo"])
    df["time"] = pd.to_datetime(df["time"])
    df["amo"] = df["amo"].replace(-99.99, np.nan)
    ds = df.set_index("time").to_xarray()
    cut_ds = ds.sel(time=slice("2002-09-01", "2025-08-31"))
    return cut_ds['amo']

In [ ]:
#Import Component Datasets

filenames = ['aao.data', 'amon.us.data', 'ao.data','enso.data','nao.data','pdo.data','qbo.data','iod.data', 'solar.data']
comp_labelz = ['aao', 'amo', 'ao', 'enso', 'nao', 'pdo', 'qbo', 'iod', 'solar']

combined = []
for i, file in enumerate(filenames):
    df = open_sesame(file).to_dataframe(name=comp_labelz[i]).reset_index()
    df = df.drop(columns=['time'])
    combined.append(df)
Components = pd.concat(combined, axis=1)
Components['amo'] = Components['amo'] - Components['amo'].mean()
Components['saod'] = xr.open_dataset(your_dir + '/Fitting Components/global_saod.nc')["Glossac_Aerosol_Optical_Depth"].to_dataframe(name='saod').reset_index()['saod']
Components['saod'] = Components['saod'] - Components['saod'].mean()
#Components['enso_sq'] = Components['enso']**2
Components = Components[0:264]
Components['trend'] = np.linspace(-131, 132, 264)
Components['sin_12'] = np.sin(2*np.pi*np.linspace(-131, 132, 264)/12)
Components['cos_12'] = np.cos(2*np.pi*np.linspace(-131, 132, 264)/12)
Components = (Components - Components.mean()) / Components.std()
Components = sm.add_constant(Components)
print(Components)

##### Parallelized Environment / OLS Regression Functions

In [ ]:
# Combined Dask setup + machine probe (set env BEFORE starting workers)
import os
import psutil
from dask.distributed import LocalCluster, Client

# Set BLAS/threads env vars via dictionary update so worker processes do not oversubscribe threads
os.environ.update({'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1',  'MKL_NUM_THREADS': '1'})

# Probe machine resources
n_logical = psutil.cpu_count(logical=True)
n_physical = psutil.cpu_count(logical=False) or 'unknown'
avail_gb = psutil.virtual_memory().available // (1024 ** 3)
print(f"CPUs: logical={n_logical}, physical={n_physical}, available RAM ≈ {avail_gb} GiB")

# Tune this value (GiB per worker). Lower => more workers but greater OOM risk. Pick 2-4 based on your workload; small single-threaded tasks can use 2-3 GiB.
desired_mem_per_worker_gb = 3

# Compute a safe suggested number of workers (cap at logical CPUs) and remove redundant boundary checking
target_workers = int(min(n_logical, max(1, avail_gb // desired_mem_per_worker_gb)))
print(f"Suggested workers (with {desired_mem_per_worker_gb} GiB each): {target_workers}")

# Reserve 4 GiB of RAM for system + scheduler, then compute per-worker limits
usable_gb = max(1, int(avail_gb - 4))
per_worker_gb = max(2, usable_gb // max(1, target_workers))
memory_limit = f"{per_worker_gb}GB"

print(f"Starting/adjusting cluster: target_workers={target_workers}, memory_limit={memory_limit}, threads_per_worker=1")

# Use a flag to deduplicate the cluster instantiation logic
create_new_cluster = True

if 'cluster' in globals() and isinstance(cluster, LocalCluster):
    try:
        cluster.scale(target_workers)
        client = Client(cluster)
        print('Scaled existing LocalCluster')
        create_new_cluster = False
    except Exception as e:
        print('Scaling existing cluster failed, creating new cluster instead:', e)

if create_new_cluster:
    cluster = LocalCluster(n_workers=target_workers, threads_per_worker=1, memory_limit=memory_limit, processes=True)
    client = Client(cluster)

print(client)
print('Cluster workers:', client.ncores())

# Quick sanity: show worker memory limits and thread count using a direct dict view
for w, v in client.scheduler_info()['workers'].items():
    print(v['name'], 'nthreads=', v['nthreads'], 'mem_limit_GB=', round(v['memory_limit'] / (1024 ** 3), 2))

# - To change the worker count later: use `cluster.scale(new_count)`
# - To stop/cleanup: call `client.close()` and `cluster.close()`

In [ ]:
# Fast block-parallel regression using NumPy/LAPACK (ChatGPT-assisted)
import numpy as np
import xarray as xr
import time
from distributed import wait
from scipy.stats import t as student_t, f as fisher_f

def align_comp_vals_to_ds(ds, comp_vals):
    """
    Ensure comp_vals has the same length as ds.time.
    Only truncate once per dataset.

    ds: xarray.DataArray with dimension 'time'
    comp_vals: np.ndarray or xr.DataArray with shape (ntime_comp, n_components)
    Returns:
        comp_vals_aligned: np.ndarray with shape (ntime_ds, n_components)
    """
    ntime_ds = ds.sizes['time']
    
    if isinstance(comp_vals, xr.DataArray):
        comp_vals = comp_vals.values
        
    ntime_comp = comp_vals.shape[0]
    
    if ntime_comp > ntime_ds:
        return comp_vals[:ntime_ds, :]
    elif ntime_comp < ntime_ds:
        raise ValueError(f"Component array shorter ({ntime_comp}) than dataset time ({ntime_ds}).")
    
    return comp_vals

# ------------------------
# Utility: safe alignment
def prepare_comp_arr_for(ds, comp_arr):
    ds_time = ds.coords['time'].values
    comp_time = comp_arr.coords['time'].values
    
    if np.array_equal(comp_time, ds_time):
        return comp_arr.sel(time=ds.time)
        
    ds_size = ds.sizes['time']
    
    # Consolidates both the exact match and greater-than size scenarios
    if comp_arr.sizes['time'] >= ds_size:
        return comp_arr.isel(time=slice(0, ds_size)).assign_coords(time=ds.coords['time'])
        
    common = np.intersect1d(comp_time, ds_time)
    if common.size == ds_size:
        return comp_arr.sel(time=common).assign_coords(time=ds.coords['time'])
        
    raise ValueError("Cannot align comp_arr.time to ds.time automatically; rebuild Components to match ds time or resample.")

# ------------------------
# Core: per-series OLS implemented with NumPy (used ChatGPT to assist with coding)
def per_series_regression_numpy(ts, comp_vals, alpha=0.05):
    """
    ts: 1D array (time,)
    comp_vals: 2D array (time, n_components)
    Returns:
      base: (7,) array in order [rsquared, rsquared_adj, aic, bic, fvalue, f_pvalue, llf]
      comps: (n_components,5) for each component [coef, std_err, pval, conf_low, conf_high]
      model_ts: (ntime,) predicted y
      resid_ts: (ntime,) residuals y - yhat
    """
    ts = np.asarray(ts).squeeze()
    if ts.ndim != 1:
        ts = ts.ravel()
    n_time = ts.shape[0]

    comp_vals = np.asarray(comp_vals)
    if comp_vals.ndim == 1:
        comp_vals = comp_vals.reshape(n_time, 1)
    elif comp_vals.ndim == 2 and comp_vals.shape[0] != n_time and comp_vals.shape[1] == n_time:
        comp_vals = comp_vals.T
    else:
        comp_vals = comp_vals.reshape(n_time, -1)

    n_components = comp_vals.shape[1]
    
    # Consolidate standard NaN returns for all invalid conditions
    nan_returns = (np.full(7, np.nan), np.full((n_components, 5), np.nan), 
                   np.full(n_time, np.nan), np.full(n_time, np.nan))

    if comp_vals.shape[0] != n_time or n_time == 0 or np.all(np.isnan(ts)):
        return nan_returns

    ts_nans = np.isnan(ts)
    if ts_nans.any():
        valid = ~ts_nans
        if valid.sum() < 2:
            return nan_returns
        ts = ts.copy()
        x = np.arange(n_time)
        ts[ts_nans] = np.interp(x[ts_nans], x[valid], ts[valid])

    # Build design matrix X = [const, comps...]
    X = np.empty((n_time, n_components + 1), dtype=float)
    X[:, 0] = 1.0
    X[:, 1:] = comp_vals.astype(float)

    try:
        beta, *_ = np.linalg.lstsq(X, ts, rcond=None)
    except Exception:
        beta = np.linalg.pinv(X) @ ts

    yhat = X @ beta
    resid = ts - yhat

    n, p = n_time, X.shape[1]
    dof = n - p
    SSR = float(np.sum(resid ** 2))
    SST = float(np.sum((ts - ts.mean()) ** 2))

    r2 = (1.0 - SSR / SST) if SST != 0 else (1.0 if SSR == 0 else np.nan)
    r2_adj = 1.0 - (1.0 - r2) * (n - 1) / dof if (dof > 0 and not np.isnan(r2)) else np.nan
    sigma2 = SSR / dof if dof > 0 else np.nan

    XtX = X.T @ X
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)

    # Direct computation of se_beta without saving unused dense covariance matrix
    if not np.isnan(sigma2):
        se_beta = np.sqrt(np.maximum(np.diag(sigma2 * XtX_inv), 0.0))
    else:
        se_beta = np.full(p, np.nan)

    with np.errstate(divide='ignore', invalid='ignore'):
        tvals = beta / se_beta
        
    pvals = 2.0 * (1.0 - student_t.cdf(np.abs(tvals), df=dof)) if dof > 0 else np.full_like(tvals, np.nan)

    if dof > 0:
        margin = student_t.ppf(1.0 - alpha / 2.0, df=dof) * se_beta
        conf_low = beta - margin
        conf_high = beta + margin
    else:
        conf_low = conf_high = np.full_like(beta, np.nan)

    num_df, den_df = p - 1, dof
    if num_df > 0 and den_df > 0:
        if (1.0 - r2) == 0:
            F, pF = np.inf, 0.0
        else:
            F = (r2 / num_df) / ((1.0 - r2) / den_df)
            pF = 1.0 - fisher_f.cdf(F, num_df, den_df)
    else:
        F = pF = np.nan

    base = np.array([r2, r2_adj, np.nan, np.nan, F, pF, np.nan], dtype=float)
    comps = np.stack([beta[1:], se_beta[1:], pvals[1:], conf_low[1:], conf_high[1:]], axis=1)

    return base, comps, yhat, resid


# ------------------------
# Worker-side block processor: uses per_series_regression_numpy
def process_block_numpy(sub_da, comp_vals):
    """
    sub_da: xarray.DataArray (time, space_block) - may be lazy; calling .values executes on worker
    comp_vals: numpy 2D array (time, n_components) - already scattered/broadcast to workers
    """
    data = sub_da.values
    if data.ndim == 1:
        data = data.reshape(-1, 1)
        
    n_time, n_space = data.shape
    n_components = int(comp_vals.shape[1])

    # Replaced np.full with np.empty since arrays are deterministically filled in loop
    base_block = np.empty((n_space, 7), dtype=float)
    comp_block = np.empty((n_space, n_components, 5), dtype=float)
    model_block = np.empty((n_space, n_time), dtype=float)
    resid_block = np.empty((n_space, n_time), dtype=float)

    for k in range(n_space):
        base, comps, yhat, resid = per_series_regression_numpy(data[:, k], comp_vals)
        base_block[k, :] = base
        
        # Eliminated redundant boolean check, direct min sizing applies cleanly to both states
        nmin = min(comps.shape[0], n_components)
        comp_block[k, :nmin, :] = comps[:nmin, :]
        
        model_block[k, :] = yhat
        resid_block[k, :] = resid

    return base_block, comp_block, model_block, resid_block

# ------------------------
# Block-parallel driver (uses client.submit on process_block_numpy, ChatGPT-assisted)
def run_block_parallel_with_comp_future_numpy(ds, comp_future, block_size=None, verbose=True,
                                              deseasonalize=False, nan_frac_thresh=0.2):
    if deseasonalize:
        # Cache the grouping locally to avoid re-evaluating the groupby twice
        gb = ds.groupby('time.month')
        ds = gb - gb.mean('time')
        ds = ds.where(ds.isnull().mean(dim='time') < nan_frac_thresh)

    ds_stacked = ds.stack(space=('lat', 'lon')).transpose('time', 'space') if 'space' not in ds.dims else ds
    nspace, ntime = int(ds_stacked.sizes['space']), int(ds_stacked.sizes['time'])

    if verbose:
        print("Aligning component matrix to dataset time...")

    comp_vals = comp_future.result() if hasattr(comp_future, "key") else comp_future
    comp_fut = client.scatter(align_comp_vals_to_ds(ds, comp_vals), broadcast=True)

    total_cores = sum(client.ncores().values())
    block_size = block_size or max(1, int(np.ceil(nspace / total_cores)))

    if verbose:
        print(f"nspace: {nspace} ntime: {ntime} block_size: {block_size} total_cores: {total_cores}")

    ds_stacked = ds_stacked.chunk({'time': -1, 'space': block_size}).persist()
    time.sleep(0.2)
    client.rebalance([ds_stacked])
    time.sleep(0.1)

    # Simplified bounds logic with list comprehension
    futures = [client.submit(process_block_numpy, ds_stacked.isel(space=slice(start, min(start + block_size, nspace))), comp_fut) for start in range(0, nspace, block_size)]

    wait(futures)
    results = client.gather(futures)

    base_all = np.vstack([r[0] for r in results])
    comp_all = np.vstack([r[1] for r in results])
    model_all = np.vstack([r[2] for r in results])
    resid_all = np.vstack([r[3] for r in results])

    # Cache coordinates to prevent multiple redundant dictionary lookups
    coords_space = ds_stacked.coords['space']
    coords_time = ds_stacked.coords['time']

    m_base = xr.DataArray(base_all.T, dims=('metric_base','space'),
                          coords={'metric_base': np.arange(base_all.shape[1]), 'space': coords_space})

    try:
        comp_coord_vals = comp_arr.coords['component'] if 'comp_arr' in globals() and hasattr(comp_arr, 'coords') and 'component' in comp_arr.coords else np.arange(comp_all.shape[1])
    except Exception:
        comp_coord_vals = np.arange(comp_all.shape[1])

    m_comp = xr.DataArray(np.transpose(comp_all, (1, 2, 0)),
                          dims=('component','metric_comp','space'),
                          coords={'component': comp_coord_vals, 'metric_comp': np.arange(5), 'space': coords_space})

    model_ts = xr.DataArray(model_all.T, dims=('time','space'), coords={'time': coords_time, 'space': coords_space})
    resid_ts = xr.DataArray(resid_all.T, dims=('time','space'), coords={'time': coords_time, 'space': coords_space})

    return m_base, m_comp, model_ts, resid_ts

##### Run Functions & Save Regression Model Parameters

In [ ]:
# Rebuild comp_arr from Components DataFrame
comp_arr = xr.DataArray(Components.values, dims=('time', 'component'),
                        coords={'time': Components.index, 'component': Components.columns})
print(f"Rebuilt comp_arr from `Components` (shape = {comp_arr.shape})")

# Ensure client exists safely
if 'client' not in globals():
    raise RuntimeError("`client` not found in this kernel — start your Dask client first")

# Cancel previous comp_future if present and purge from globals
if 'comp_future' in globals():
    try:
        client.cancel(comp_future)
        print("Cancelled previous comp_future on scheduler/workers.")
    except Exception as e:
        print("Warning: failed to cancel previous comp_future:", e)
    globals().pop('comp_future', None)

# Scatter new comp_vals (broadcast to all workers)
comp_vals = comp_arr.values
t0 = time.time()
comp_future = client.scatter(comp_vals, broadcast=True)
print(f"Scattered new comp_vals (broadcast) in {time.time() - t0:.3f} s; comp_vals.shape = {comp_vals.shape}")

# Ensure model_arr exists using a concise list comprehension
if 'model_arr' not in globals():
    model_arr = [np.empty(10, dtype=object) for _ in range(4)]
    print("Created fallback model_arr")

results = {}
j_start, j_stop = 0, 1
i_start, i_stop = 0, 9  # adjust as needed

for j in range(j_start, j_stop):
    for i in range(i_start, i_stop):
        print(f"\n--- Processing var_arr[{j}][{i}] ---")
        
        # Consolidate invalid array states to a single failure path
        try:
            ds = var_arr[j][i]
            if ds is None: 
                raise ValueError("Dataset is None")
        except Exception:
            print(" Dataset not accessible or None; skipping")
            results[(j, i)] = (None, None, None, None)
            continue

        t0 = time.time()
        try:
            # --- USE THE NEW NUMPY/LAPACK FUNCTION ---
            res = run_block_parallel_with_comp_future_numpy(ds, comp_future, block_size=8192, verbose=True,  deseasonalize=True, nan_frac_thresh=0.2)
        except Exception:
            import traceback
            traceback.print_exc()
            results[(j, i)] = (None, None, None, None)
            continue
            
        m_base, m_comp, model_ts, resid_ts = res
        print(f" Completed in {time.time()-t0:.2f}s; shapes: m_base {getattr(m_base,'shape',None)}, m_comp {getattr(m_comp,'shape',None)}")

        # Store outputs globally and in the mapping
        results[(j, i)] = res

        try:
            # Dynamic fallback index handling rather than duplicative logic structures
            target_j = j if j < len(model_arr) else 0
            model_arr[target_j][i] = res
            print(f" Stored into model_arr[{target_j}][{i}] ({'direct by j' if target_j == j else 'fallback index 0'})")
        except Exception:
            print(" Could not store into model_arr; result kept in `results` mapping")

print("\nDone. Inspect `model_arr` and `results` for outputs.")

In [ ]:
# Safe saving of model_arr entries (unstack fallback + compression)
import os
import pickle
import numpy as np
import xarray as xr

# Move helper function OUTSIDE the loop to avoid redundant redefinition
def unstack_or_reshape(da, name, ds, reshape_dims):
    if da is None:
        return None
    try:     # Try unstack first
        if 'space' in da.dims:
            return da.unstack('space')
        return da
    except Exception:
        pass
        
    # If unstack fails, reshape numerically
    if ds is None:
        return da  # fallback: leave as-is
        
    try:
        dims = reshape_dims
        shape = tuple(np.prod([ds.sizes[d] for d in dims], dtype=int) if d in ds.sizes else da.sizes[d] for d in dims)
        # Fix: used 'idx' instead of 'i' to avoid variable shadowing with the outer loop
        coords = {d: ds.coords[d] if d in ds.coords else np.arange(shape[idx]) for idx, d in enumerate(dims)}
        vals = np.asarray(da.values).reshape(shape)
        return xr.DataArray(vals, dims=dims, coords=coords)
    except Exception:
        print(f"Could not reshape {name}; keeping original")
        return da

sourcedatasite = ['AIRSv7', 'CLIMCAPS', 'MERRA2', 'ERA5', 'JRA55', 'BE', 'JRA3Q']
filelabels = ['ST', 'SH', '850T', '500T']

out_dir = your_dir + '/Comparisons/ST'
os.makedirs(out_dir, exist_ok=True)

for j in range(0, 1):
    for i in range(4, 5):
        entry = model_arr[j][i]
        if entry is None:
            print(f"Skipping model_arr[{j}][{i}] (None)")
            continue
            
        m_base, m_comp, model_ts, resid_ts = entry

        try:         # original dataset used to make this result (needed for lat/lon sizes/coords)
            ds = var_arr[j][i]
        except Exception:
            ds = None

        # Group metadata to process and save arrays systematically
        datasets_to_process = [
            (m_base, 'm_base', 'base', ['metric_base', 'lat', 'lon']),
            (m_comp, 'm_comp', 'comp', ['component', 'metric_comp', 'lat', 'lon']),
            (model_ts, 'model_ts', 'model_ts', ['time', 'lat', 'lon']),
            (resid_ts, 'resid_ts', 'resid_ts', ['time', 'lat', 'lon'])
        ]

        # Process and save each dataset sequentially
        for da, name, suffix, dims in datasets_to_process:
            da_un = unstack_or_reshape(da, name, ds, dims)
            fname = os.path.join(out_dir, f"{sourcedatasite[i]}_{filelabels[j]}_{suffix}.nc")
            
            try:
                da_un.to_netcdf(fname)
                print(f"Saved {fname}")
            except Exception as e:
                # fallback: pickle
                pk_file = fname.replace('.nc', '.pkl')
                with open(pk_file, 'wb') as f:
                    pickle.dump(da_un, f)
                print(f"NetCDF save failed; wrote pickle {pk_file}. Error: {e}")

##### Load in (i.e. from previous session)

In [ ]:
modelz_surftemp = np.empty(10, dtype='object')
modelz_surfhum = np.empty(10, dtype='object')
modelz_850temp = np.empty(10, dtype='object')
modelz_500temp = np.empty(10, dtype='object')
model_arr = [modelz_surftemp, modelz_surfhum, modelz_850temp, modelz_500temp]

In [ ]:
# Helper cell: robust loader that converts 'space' -> (lat,lon)
import os
import xarray as xr
import numpy as np
import pandas as pd

def _try_unstack_space(da):
    if 'space' not in da.dims:
        return da
    try:
        return da.unstack('space')
    except Exception:
        return None

def _coords_from_da_space_coords(da):
    """If da has coord arrays that encode lat/lon per space entry, return lat, lon arrays or None."""
    # common coord names
    possible_lat_names = ['lat', 'latitude', 'space_lat', 'space_latitude']
    possible_lon_names = ['lon', 'longitude', 'space_lon', 'space_longitude']
    lat_coord = None
    lon_coord = None
    for n in possible_lat_names:
        if n in da.coords:
            arr = da.coords[n]
            if arr.shape and arr.shape[0] == da.sizes['space']:
                lat_coord = arr
                break
    for n in possible_lon_names:
        if n in da.coords:
            arr = da.coords[n]
            if arr.shape and arr.shape[0] == da.sizes['space']:
                lon_coord = arr
                break
    if lat_coord is not None and lon_coord is not None:
        return lat_coord, lon_coord
    return None

def _reshape_using_ref(da, ref_ds):
    """Reshape da.values using ref_ds lat/lon sizes and coords."""
    if ref_ds is None:
        return None
    if 'lat' not in ref_ds.dims or 'lon' not in ref_ds.dims:
        return None
    nlat = int(ref_ds.sizes['lat'])
    nlon = int(ref_ds.sizes['lon'])
    space_len = da.sizes.get('space', None)
    if space_len is None or space_len != nlat * nlon:
        return None
    vals = np.asarray(da.values)
    # handle cases: (metric_base, space) or (space,) or (component, metric_comp, space)
    if vals.ndim == 1:
        vals2 = vals.reshape((nlat, nlon))
        dims = ('lat','lon')
        coords = {'lat': ref_ds.coords['lat'], 'lon': ref_ds.coords['lon']}
        return xr.DataArray(vals2, dims=dims, coords=coords)
    elif vals.ndim == 2:
        # assume (X, space) -> (X, lat, lon)
        vals2 = vals.reshape((vals.shape[0], nlat, nlon))
        dims = (da.dims[0], 'lat', 'lon')
        coords = {da.dims[0]: da.coords.get(da.dims[0], np.arange(vals2.shape[0])),
                  'lat': ref_ds.coords['lat'], 'lon': ref_ds.coords['lon']}
        return xr.DataArray(vals2, dims=dims, coords=coords)
    elif vals.ndim == 3:
        # assume (component, metric_comp, space) -> (component, metric_comp, lat, lon)
        vals2 = vals.reshape((vals.shape[0], vals.shape[1], nlat, nlon))
        dims = (da.dims[0], da.dims[1], 'lat', 'lon')
        coords = {da.dims[0]: da.coords.get(da.dims[0], np.arange(vals2.shape[0])),
                  da.dims[1]: da.coords.get(da.dims[1], np.arange(vals2.shape[1])),
                  'lat': ref_ds.coords['lat'], 'lon': ref_ds.coords['lon']}
        return xr.DataArray(vals2, dims=dims, coords=coords)
    else:
        return None

def ensure_latlon_da(da, ref_ds=None):
    """
    Ensure DataArray `da` has lat/lon dims. Returns a DataArray with lat/lon dims or raises.
    ref_ds: optional reference dataset (e.g. var_arr[j][i]) to recover lat/lon sizes/coords.
    """
    # 1) already has lat/lon?
    if ('lat' in da.dims and 'lon' in da.dims) or ('latitude' in da.dims and 'longitude' in da.dims):
        return da

    # 2) try unstack if 'space' MultiIndex present
    if 'space' in da.dims:
        got = _try_unstack_space(da)
        if got is not None:
            # after unstack, dims might be ('metric_base','lat','lon') or ('component','metric_comp','lat','lon')
            return got

        # 3) try to extract lat/lon coords attached per space entry
        latlon = _coords_from_da_space_coords(da)
        if latlon is not None:
            lat_coord, lon_coord = latlon
            # build MultiIndex and unstack
            try:
                mi = pd.MultiIndex.from_arrays([lat_coord.values, lon_coord.values], names=('lat','lon'))
                da2 = da.assign_coords(space=mi)
                return da2.unstack('space')
            except Exception:
                pass

        # 4) try reshape using ref_ds
        if ref_ds is not None:
            got2 = _reshape_using_ref(da, ref_ds)
            if got2 is not None:
                return got2

    # 5) as a last attempt, if da has numeric 'space' coords that are indices and ref_ds provides lat/lon coordinates
    if 'space' in da.dims and ref_ds is not None:
        try:
            # Try mapping integer space index -> lat/lon via a simple reshape
            return _reshape_using_ref(da, ref_ds)
        except Exception:
            pass

    # cannot convert automatically
    raise RuntimeError("Cannot convert DataArray with dims %s to lat/lon. Provide `ref_ds` or save with lat/lon dims originally." % (da.dims,))

# Wrapper to open files and convert
def load_model_quad(base_file, comp_file, model_ts_file, resid_ts_file, ref_ds=None):

    def _open_first_var(fname):
        ds = xr.open_dataset(fname)
        return ds[list(ds.data_vars)[0]]

    # --- spatial fields ---
    da_base = _open_first_var(base_file)
    da_comp = _open_first_var(comp_file)

    da_base = ensure_latlon_da(da_base, ref_ds=ref_ds)
    da_comp = ensure_latlon_da(da_comp, ref_ds=ref_ds)

    # --- time series (NO spatial conversion) ---
    model_ts = _open_first_var(model_ts_file)
    resid_ts = _open_first_var(resid_ts_file)

    return da_base, da_comp, model_ts, resid_ts

In [ ]:
sourcedatasite = ['AIRSv7', 'CLIMCAPS', 'MERRA2', 'ERA5', 'JRA55',  'JRA3Q', 'BEST', 'GISTEMP','AIRSv7Skin']
filelabels = ['ST', 'SH', '850T', '500T']
path = your_dir + '/data/Comparisons'

for j in range(0,1):
    for i in range(9):
        folder = os.path.join(path, filelabels[j])
        base_file = os.path.join(folder, f'{sourcedatasite[i]}_{filelabels[j]}_base.nc')
        comp_file = os.path.join(folder, f'{sourcedatasite[i]}_{filelabels[j]}_comp.nc')
        model_ts_file = os.path.join(folder, f'{sourcedatasite[i]}_{filelabels[j]}_model_ts.nc')
        resid_ts_file = os.path.join(folder, f'{sourcedatasite[i]}_{filelabels[j]}_resid_ts.nc')
        if not (os.path.exists(base_file) and os.path.exists(comp_file)):
            print("Missing:", base_file, comp_file)
            model_arr[j][i] = None
            continue
        # pass ref_ds so reshape can use original coords (if available)
        ref_ds = None
        try:
            ref_ds = var_arr[j][i]
        except Exception:
            ref_ds = None

        try:
            da_base_fixed, da_comp_fixed, model_ts, resid_ts = load_model_quad(base_file, comp_file, model_ts_file, resid_ts_file, ref_ds=ref_ds)
            model_arr[j][i] = (da_base_fixed, da_comp_fixed, model_ts, resid_ts)
            print(f"Loaded and fixed model_arr[{j}][{i}]: base {da_base_fixed.dims}, comp {da_comp_fixed.dims}")
        except Exception as e:
            print(f"Failed to load/convert for {i},{j}: {e}")
            model_arr[j][i] = None

def normalize_lon(da):
    da = da.assign_coords(lon=((da.lon + 180) % 360) - 180)
    da = da.sortby("lon")
    return da

for i in [4, 6]:
    jra55, jra51, model_ts, resid_ts = model_arr[0][i]
    jra51, jra55, model_ts, resid_ts = normalize_lon(jra51), normalize_lon(jra55), normalize_lon(model_ts), normalize_lon(resid_ts)
    model_arr[0][i] = (jra55, jra51, model_ts, resid_ts)

##### Plotting

In [ ]:
# Plots the Global Gridded Trend Values / Confidence for all Datasets Together

sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA', 'MERRA2',  'ERA5', 'JRA55', 'JRA3Q','BEST','GISTEMP', 'AIRSv7 Skin']
filelabels = ['ST', 'SH', '850T', '500T']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['°K/decade', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST_comp_trend','SH_comp_trend','850T_comp_trend','500T_comp_trend']
vextremas = [1, 0.001*1000, 1, 1]
colormapz = ['seismic', 'BrBG', 'seismic', 'seismic']

trend_tot = np.empty((4, 18), dtype='object')
plt.close('all')

for j in range(0,1):  # loop over variables
    trend_vals = []
    pval_vals = []

    for i in [0,1,2,3,4,5,6,7,8,9]:  # loop over datasets
        model_data = model_arr[j][i]
        if model_data is None or model_data[0] is None:
            trend_vals.append(None)
            pval_vals.append(None)
            continue

        b_A7ST, c_A7ST, d, e = model_data

        # Trend map (metric_comp=0)
        trend_map = c_A7ST.isel(component=11).isel(metric_comp=0)
        if j == 1:
            trend_map = c_A7ST.isel(component=11).isel(metric_comp=0)
        trend_vals.append(trend_map)

        # P-value map (metric_comp=2)
        pval_map = c_A7ST.isel(component=11).isel(metric_comp=2)
        pval_vals.append(pval_map)

        # Store in trend_tot if needed
        trend_tot[j][2*i] = trend_map
        trend_tot[j][2*i+1] = pval_map
    #trend_vals.append(None)  # for the 8th slot (if needed)
    #pval_vals.append(None)  # for the 8th slot (if needed)
    # 6-panel plotting
    SixPlotter67(
        temp=trend_vals,
        cmap=colormapz[j],
        suptitle='Global Gridded '+varlabels[j]+' Trend (09-2002 to 08-2023)',
        title=sourcelabels[:],
        vmin=-vextremas[j],
        vmax=vextremas[j],
        filename=filelabels[j]+'.png',
        variable_name='Trend ['+unitlabels[j]+']',
        pvals=pval_vals,
        significance=0.05
    )


In [ ]:
# Plots the Coefficients, R^2, and F-Test p-value over the global grid for a single dataset, with differing saves for all datasets

unitsz = ['°K/', 'g/kg ', '°K/', '°K/']
coladosad = ['seismic', 'BrBG' , 'seismic', 'seismic']
filelabels=['ST','SH','850T','500T']
sourcedatasite = ['AIRSv7', 'CLIMCAPS-Aqua', 'MERRA2', 'ERA5', 'JRA55',  'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']

for j in range(0,1):
    for i in range(8,9):
        b_A7ST, c_A7ST, d, e = model_arr[j][i]
        if b_A7ST is None:
            continue
        sourcelabels = ['$R^2$', 'F p-value', 'Trend', 'AAO', 'AMO', 'AO', 'ENSO', 'NAO', 'PDO', 'PNA', 'QBO', 'IOD', 'Solar Cycle', 'SAOD', 'Annual Sine', 'Annual Cosine']
        modez_title = sourcedatasite[i]+' Global Gridded SAT Anomaly OLS Multiple Linear Regression'
        colormapzsz = ['Greens','PiYG_r', coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j], coladosad[j]]
        vmins = [0, 0, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5]
        vmaxs = [1, 0.1, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5]
        b_plotts = [b_A7ST.isel(metric_base=0), b_A7ST.isel(metric_base=5)]
        c_plotts = [c_A7ST.sel(component=comp).isel(metric_comp=0) for comp in c_A7ST['component'].values]
        p_plotts = [c_A7ST.sel(component=comp).isel(metric_comp=2) for comp in c_A7ST['component'].values]
        das = np.empty(2, dtype='object')
        for k, da in enumerate([c_plotts, p_plotts]):
            thetrend = da[-3]
            thepna = da[-4]
            das[k] = [thetrend] + da[0:6] + [thepna] + da[6:10]+ da[-2:]
        c_plotts, p_plotts = das
        p_plotts = [None, b_A7ST.isel(metric_base=5)] + p_plotts
        modez_plotts = b_plotts + c_plotts
        for da in modez_plotts:
            if da is not None:
                da.name = 'Regression Coefficient ['+unitsz[j]+'decade]'
        SixteenPlotter(
            temp=modez_plotts,
            pvals=p_plotts,                     # or list of 16 p-value DataArrays
            cmapz=colormapzsz,
            suptitle=modez_title,
            title=sourcelabels,             # now used properly
            vmins=vmins,
            vmaxs=vmaxs,
            filename='Coefficient_Maps_'+filelabels[j]+'_'+sourcedatasite[i]+'.png',
            variable_name='Coefficients ['+unitsz[j]+'decade]',
            significance=0.05               # optional
        )

plt.close()

### Global / Zonal / Regional Means (for subsequent time series analysis)

Make sure that the variables are all loaded!

In [ ]:
from dask import delayed

#land/sea masks
from regionmask.defined_regions import natural_earth_v5_0_0 as ne
land = ne.land_110
mask = land.mask(var_arr[0][0]['lon'], var_arr[0][0]['lat'])
land_mask = mask.where(mask > 0.5)   # land
ocean_mask = mask.where(mask < 0.5)  # ocean

def region_selector(var_prev, reg='global'):
    if reg == 'global':
        return var_prev
    elif reg == '60N-60S':
        var_prev = var_prev.assign_coords(lon=(((var_prev['lon'] + 180) % 360) - 180))
        var_prev = var_prev.sortby('lon')  # ensure ascending order
        var_z = var_prev.sel(lat=slice(-60, 60))
        return var_z
    elif reg == 'SubSaharanAfrica':
        var_prev = var_prev.assign_coords(lon=(((var_prev['lon'] + 180) % 360) - 180))
        var_prev = var_prev.sortby('lon')  # ensure ascending order
        var_prev = var_prev.sel(lat=slice(-15, 15), lon=slice(-19,52))
        var_z = var_prev.where(land_mask.broadcast_like(var_prev))
        return var_z
    elif reg == "SouthernOcean":
        var_prev = var_prev.assign_coords(lon=(((var_prev['lon'] + 180) % 360) - 180))
        var_prev = var_prev.sortby('lon')  # ensure ascending order
        var_prev = var_prev.sel(lat=slice(-90, -60))
        var_z = var_prev.where(ocean_mask.broadcast_like(var_prev))
        return var_z
    elif reg == "LandOnly":
        var_prev = var_prev.assign_coords(lon=(((var_prev['lon'] + 180) % 360) - 180))
        var_prev = var_prev.sortby('lon')  # ensure ascending order
        var_z = var_prev.where(land_mask.broadcast_like(var_prev))
        return var_z

# Pre-allocate output
des_glob_meanz = np.empty((4, 11), dtype='object')

In [ ]:
import time
from dask.diagnostics import ProgressBar

# Weighted mean computation (LAZY)
def compute_weighted_mean(var_z, j, i, region='global'):
    """
    Compute weighted global mean anomaly for one dataset.
    Returns a Dask-backed xarray object.
    """
    if var_z is None:
        return j, i, None

    # Sorting
    var_z = var_z.sortby('lat').sortby('lon')
    # Region selection
    var_z = region_selector(var_z, region)
    # Latitude weights
    weights = np.cos(np.deg2rad(var_z['lat']))
    # Global weighted mean
    result = var_z.weighted(weights).mean(dim=('lat', 'lon'))

    return j, i, result

In [ ]:
# Execution
print("Running 9 datasets with Dask-only execution...")
t0 = time.time()

# Chunk EACH dataset (var_arr is an object array)
for idy in range(0,1):
    for idx in range(0,9):
        if var_arr[idy][idx] is not None:
            var_arr[idy][idx] = var_arr[idy][idx].chunk({
                'time': 120,
                'lat': 90,
                'lon': 90,
            })

# Build tasks (no computation yet)
tasks = [compute_weighted_mean(var_arr[j][i], j, i, region='60N-60S') for i in range(0,9) for j in range(0,1) ]

# Compute everything ONCE
with ProgressBar():
    results = dask.compute(*tasks)

# Store results
for j_out, i_out, result in results:
    des_glob_meanz[j_out][i_out] = result
    if result is not None:
        print(f"✓ j={j_out} i={i_out}: shape {result.shape}, type {type(result.data)}")
    else:
        print(f"✗ j={j_out} i={i_out}: None")

elapsed = time.time() - t0
print(f"Total time: {elapsed:.1f}s")

In [ ]:
# Save (no compute needed - results are already materialized)
output_dir = your_dir + "/data/Means/60N-60S"
os.makedirs(output_dir, exist_ok=True)
4
print("Saving...")
for j in range(0, 1):
    for i in range(0, 9):
        da = des_glob_meanz[j][i]
        if da is None:
            print("None")
            continue
        filename = f"glob_mean_var{j}_ds{i}.nc"
        da.to_netcdf(os.path.join(output_dir, filename))
        print(f"✓ Saved j={j} i={i}")

print("All files saved.")